# Day 19 — Agent concepts: tools, the loop, and guards

An **agent** is an LLM in a loop: it picks an action (usually a tool call), you run it, feed
the result back, and repeat until it produces a final answer. We build that loop from scratch,
run it on a multi-step task, then break it and add the guards every real agent needs.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | Prompt-response vs agent | 4 min |
| 1 | Tools: a schema and a registry | 8 min |
| 2 | The loop: think → act → observe → repeat | 14 min |
| 3 | A real multi-step task | 10 min |
| 4 | Failure modes: loops, wrong actions, runaway cost | 12 min |
| 5 | Guards, and the real Anthropic tool-use API | 9 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import json, time, re
rng_seed = 0
print("ready")

ready


## 0 — Prompt-response vs agent (4 min)

| | Prompt-response | Agent |
| --- | --- | --- |
| Control flow | fixed: one call in, one answer out | **model-driven**: model decides the next step |
| Steps | 1 | 1..N (until the model says "done" or a guard fires) |
| External actions | none (or a fixed pipeline you wrote) | tool calls the model chooses |
| Failure shape | a bad answer | a bad answer, an infinite loop, or a runaway bill |

RAG (Week 6) is a *fixed pipeline*: always retrieve once, then generate. An **agentic** version
would let the model decide whether to retrieve, how many times, and with what query. That
flexibility is the whole point — and the whole risk.

## 1 — Tools (8 min)

A tool is a function plus a JSON schema describing its name, purpose, and parameters. The
schema is what the model sees; the function is what you run.

In [2]:
# A tiny company-data world for our tools to act on.
TEAMS = {"engineering": dict(headcount=42, monthly_budget=520_000),
         "design": dict(headcount=9, monthly_budget=110_000),
         "sales": dict(headcount=18, monthly_budget=240_000)}
CASH_ON_HAND = 6_400_000
MONTHLY_REVENUE = 300_000

TOOLS = {}
def tool(name, description, parameters, destructive=False):
    def wrap(fn):
        TOOLS[name] = dict(fn=fn, destructive=destructive,
                           schema=dict(name=name, description=description,
                                       input_schema=dict(type="object", properties=parameters,
                                                         required=list(parameters))))
        return fn
    return wrap

@tool("get_team", "Look up a team's headcount and monthly budget.",
      {"team": {"type": "string", "enum": list(TEAMS)}})
def get_team(team):
    if team not in TEAMS: return {"error": f"unknown team '{team}'"}
    return TEAMS[team]

@tool("get_finance", "Get a company-level finance figure.",
      {"metric": {"type": "string", "enum": ["cash_on_hand", "monthly_revenue"]}})
def get_finance(metric):
    return {"cash_on_hand": CASH_ON_HAND, "monthly_revenue": MONTHLY_REVENUE}[metric]

@tool("calculator", "Evaluate a basic arithmetic expression, e.g. '520000 + 110000'.",
      {"expression": {"type": "string"}})
def calculator(expression):
    if not re.fullmatch(r"[\d\s.+\-*/()]+", expression): return {"error": "invalid characters"}
    try: return {"result": eval(expression, {"__builtins__": {}})}
    except Exception as e: return {"error": str(e)}

print(json.dumps([t["schema"] for t in TOOLS.values()], indent=1)[:600], "...")

[
 {
  "name": "get_team",
  "description": "Look up a team's headcount and monthly budget.",
  "input_schema": {
   "type": "object",
   "properties": {
    "team": {
     "type": "string",
     "enum": [
      "engineering",
      "design",
      "sales"
     ]
    }
   },
   "required": [
    "team"
   ]
  }
 },
 {
  "name": "get_finance",
  "description": "Get a company-level finance figure.",
  "input_schema": {
   "type": "object",
   "properties": {
    "metric": {
     "type": "string",
     "enum": [
      "cash_on_hand",
      "monthly_revenue"
     ]
    }
   },
   "required": [
    ...


## 2 — The loop (14 min)

The agent loop:

```
messages = [system, user]
loop:
    step = llm(messages, tools)          # model returns: a tool call, OR a final answer
    if step is final answer: return it
    result = run_tool(step.name, step.args)
    messages += [assistant: step, tool: result]
```

Real LLMs return **tool calls** as structured output (Anthropic `tool_use` blocks, §5). To
build the mechanics offline we use a **scripted planner** as the "LLM" — a function that,
given the conversation so far, returns the next action. It's deterministic, but the loop
around it is exactly the real one.

In [3]:
class Action:  # what the "LLM" returns each step
    def __init__(self, kind, **kw): self.kind = kind; self.__dict__.update(kw)

def run_agent(user_goal, planner, max_steps=8, verbose=True):
    messages = [{"role": "user", "content": user_goal}]
    trace = []
    for step in range(max_steps):
        act = planner(messages)
        if act.kind == "final":
            trace.append(("final", act.text))
            if verbose: print(f"[{step}] FINAL: {act.text}")
            return act.text, trace
        # execute the tool
        result = TOOLS[act.name]["fn"](**act.args) if act.name in TOOLS else {"error": "no such tool"}
        messages.append({"role": "assistant", "content": f"call {act.name}({act.args})"})
        messages.append({"role": "tool", "content": json.dumps(result)})
        trace.append((act.name, act.args, result))
        if verbose: print(f"[{step}] {act.name}({act.args}) -> {result}")
    return "STOPPED: hit max_steps", trace

In [4]:
# A scripted planner for: "What is the combined monthly budget of engineering and design,
# and how many months of runway does the company have at the current net burn?"
def budget_runway_planner(messages):
    seen = [m["content"] for m in messages if m["role"] == "tool"]
    n = len(seen)
    if n == 0: return Action("tool", name="get_team", args={"team": "engineering"})
    if n == 1: return Action("tool", name="get_team", args={"team": "design"})
    if n == 2: return Action("tool", name="calculator", args={"expression": "520000 + 110000"})
    if n == 3: return Action("tool", name="get_finance", args={"metric": "cash_on_hand"})
    if n == 4: return Action("tool", name="get_finance", args={"metric": "monthly_revenue"})
    if n == 5:
        # burn = combined team budget - revenue ; runway = cash / burn
        return Action("tool", name="calculator", args={"expression": "6400000 / (630000 - 300000)"})
    return Action("final", text="Combined eng+design budget is $630,000/month. Net burn is "
                  "$330,000/month, giving about 19.4 months of runway.")

ans, trace = run_agent(
    "Combined monthly budget of engineering and design, and months of runway at current burn?",
    budget_runway_planner)

[0] get_team({'team': 'engineering'}) -> {'headcount': 42, 'monthly_budget': 520000}
[1] get_team({'team': 'design'}) -> {'headcount': 9, 'monthly_budget': 110000}
[2] calculator({'expression': '520000 + 110000'}) -> {'result': 630000}
[3] get_finance({'metric': 'cash_on_hand'}) -> 6400000
[4] get_finance({'metric': 'monthly_revenue'}) -> 300000
[5] calculator({'expression': '6400000 / (630000 - 300000)'}) -> {'result': 19.393939393939394}
[6] FINAL: Combined eng+design budget is $630,000/month. Net burn is $330,000/month, giving about 19.4 months of runway.


Six tool calls, each result feeding the next decision — that's **multi-step reasoning through
the environment**. A single prompt could not do the arithmetic reliably (Day 05) *and* had no
way to look up the numbers. The loop gives the model both.

## 3 — A real multi-step task: the ReAct shape (10 min)

Production agents interleave a **reasoning trace** with actions ("ReAct" = Reason + Act). The
model writes a thought, then an action; the observation comes back; it thinks again. Let's
make our planner emit thoughts too, and handle a task where the *path* isn't fixed.

In [5]:
def react_planner(messages):
    tool_results = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
    goal = messages[0]["content"].lower()

    # "which team has the highest budget per person?"
    if "per person" in goal or "per head" in goal:
        looked_up = [m for m in messages if m["role"] == "assistant" and "get_team" in m["content"]]
        teams = list(TEAMS)
        if len(looked_up) < len(teams):
            t = teams[len(looked_up)]
            return Action("tool", name="get_team", args={"team": t},
                          thought=f"I need each team's budget and headcount; looking up {t}.")
        # have all three -> compute
        ratios = {teams[i]: r["monthly_budget"] / r["headcount"]
                  for i, r in enumerate(tool_results)}
        best = max(ratios, key=ratios.get)
        return Action("final", text=f"{best} has the highest budget per person "
                      f"(${ratios[best]:,.0f}/person). Full ranking: "
                      + ", ".join(f"{k} ${v:,.0f}" for k, v in sorted(ratios.items(), key=lambda x:-x[1])))
    return Action("final", text="I don't know how to approach this goal.")

ans, trace = run_agent("Which team has the highest budget per person?", react_planner)
print("\nthoughts along the way:")
for m in trace:
    pass  # thoughts are on the Action; shown inline above in a real system

[0] get_team({'team': 'engineering'}) -> {'headcount': 42, 'monthly_budget': 520000}
[1] get_team({'team': 'design'}) -> {'headcount': 9, 'monthly_budget': 110000}
[2] get_team({'team': 'sales'}) -> {'headcount': 18, 'monthly_budget': 240000}
[3] FINAL: sales has the highest budget per person ($13,333/person). Full ranking: sales $13,333, engineering $12,381, design $12,222

thoughts along the way:


The planner here **branches on the goal** and **loops a variable number of times** (once per
team). That variability is what separates an agent from a chain: a chain has a fixed number of
steps decided by you; an agent decides at runtime.

## 4 — Failure modes (12 min)

### 4a — Infinite / repeated loops

In [6]:
def stuck_planner(messages):
    # a buggy "model" that keeps re-checking the same thing and never concludes
    return Action("tool", name="get_finance", args={"metric": "cash_on_hand"})

ans, trace = run_agent("How much runway do we have?", stuck_planner, max_steps=5, verbose=False)
print("result:", ans)
print("steps taken:", len(trace), "— all identical:",
      len({(s[0], json.dumps(s[1])) for s in trace if s[0] != 'final'}) == 1)

result: STOPPED: hit max_steps
steps taken: 5 — all identical: True


In [7]:
# Loop detection: stop if the same (tool, args) repeats, or no progress is made.
def run_agent_guarded(user_goal, planner, max_steps=8, max_repeats=2, budget_calls=6, verbose=True):
    messages = [{"role": "user", "content": user_goal}]
    trace, seen, calls = [], {}, 0
    for step in range(max_steps):
        act = planner(messages)
        if act.kind == "final":
            return act.text, trace
        key = (act.name, json.dumps(act.args, sort_keys=True))
        seen[key] = seen.get(key, 0) + 1
        if seen[key] > max_repeats:
            return f"ABORTED: repeated action {key} {seen[key]}x (likely a loop)", trace
        calls += 1
        if calls > budget_calls:
            return f"ABORTED: exceeded tool-call budget ({budget_calls})", trace
        result = TOOLS[act.name]["fn"](**act.args) if act.name in TOOLS else {"error": "no such tool"}
        messages += [{"role": "assistant", "content": f"call {act.name}({act.args})"},
                     {"role": "tool", "content": json.dumps(result)}]
        trace.append((act.name, act.args, result))
        if verbose: print(f"[{step}] {act.name}({act.args}) -> {result}")
    return "STOPPED: max_steps", trace

print(run_agent_guarded("How much runway?", stuck_planner, verbose=False)[0])

ABORTED: repeated action ('get_finance', '{"metric": "cash_on_hand"}') 3x (likely a loop)


### 4b — Wrong actions (bad args, wrong tool, destructive calls)

In [8]:
def sloppy_planner(messages):
    n = len([m for m in messages if m["role"] == "tool"])
    if n == 0: return Action("tool", name="get_team", args={"team": "marketing"})   # doesn't exist
    if n == 1: return Action("tool", name="calculator", args={"expression": "import os"})  # injection attempt
    return Action("final", text="done")

_, tr = run_agent_guarded("look up marketing budget", sloppy_planner, verbose=True)
print("\n-> the tool functions themselves validated input: unknown team -> {'error': ...},")
print("   non-arithmetic expression -> rejected. NEVER trust tool arguments from the model.")

[0] get_team({'team': 'marketing'}) -> {'error': "unknown team 'marketing'"}
[1] calculator({'expression': 'import os'}) -> {'error': 'invalid characters'}

-> the tool functions themselves validated input: unknown team -> {'error': ...},
   non-arithmetic expression -> rejected. NEVER trust tool arguments from the model.


### 4c — Runaway cost

Every step is an LLM call (input grows each turn as the transcript accumulates — Day 04).
A 15-step agent on a long transcript can cost 50× a single answer. The `budget_calls` guard
and a token ceiling are not optional in production.

In [9]:
# rough cost model for an agent run
def agent_cost(steps, base_prompt_tok=800, growth_per_step=250, out_tok=120,
               pin=3/1e6, pout=15/1e6):
    total = 0.0
    for s in range(steps):
        in_tok = base_prompt_tok + growth_per_step * s
        total += in_tok*pin + out_tok*pout
    return total
for s in [1, 3, 6, 10, 20]:
    print(f"{s:2d} steps -> ${agent_cost(s):.4f}  ({agent_cost(s)/agent_cost(1):.0f}x a 1-step answer)")

 1 steps -> $0.0042  (1x a 1-step answer)
 3 steps -> $0.0149  (4x a 1-step answer)
 6 steps -> $0.0364  (9x a 1-step answer)
10 steps -> $0.0757  (18x a 1-step answer)
20 steps -> $0.2265  (54x a 1-step answer)


## 5 — Guards + the real API (9 min)

### The guard checklist

| Guard | Why |
| ----- | --- |
| `max_steps` | hard stop on iteration count |
| repeated-action / no-progress detection | catch loops the model can't see it's in |
| tool-call budget + token budget | cap the bill |
| input validation *inside every tool* | the model's args are untrusted input |
| allowlist of tools per agent | least privilege |
| human approval for destructive/irreversible actions | delete, send email, pay, deploy |
| timeout per tool call | a hanging tool shouldn't hang the agent |
| structured logging of every (thought, action, observation) | you cannot debug what you didn't trace |

### The real Anthropic tool-use loop

```python
import anthropic
client = anthropic.Anthropic()
tools = [t["schema"] for t in TOOLS.values()]          # same schemas we built
messages = [{"role": "user", "content": goal}]

for _ in range(MAX_STEPS):
    resp = client.messages.create(model="claude-opus-5", max_tokens=1024,
                                  system=SYSTEM, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": resp.content})
    if resp.stop_reason != "tool_use":
        break                                          # final answer is in resp.content text blocks
    tool_results = []
    for block in resp.content:
        if block.type == "tool_use":
            out = TOOLS[block.name]["fn"](**block.input)     # validate inside the fn!
            tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                 "content": json.dumps(out)})
    messages.append({"role": "user", "content": tool_results})   # all results in ONE message
```

Key points: the SDK gives you `tool_use` blocks (no parsing), you return **all** `tool_result`
blocks in a single `user` message, and you own the loop and its guards. The SDK's
`client.beta.messages.tool_runner` automates the loop while keeping per-turn hooks for
approval and logging.

In [10]:
# our loop, hardened, matching the shape above
SYSTEM_AGENT = ("You are a financial analyst agent. Use tools to look up figures; use the "
                "calculator for arithmetic. When you have the answer, state it with the numbers.")

def run_agent_final(goal, planner, max_steps=8, on_destructive=None):
    messages = [{"role": "user", "content": goal}]
    log = []
    for step in range(max_steps):
        act = planner(messages)
        log.append(dict(step=step, kind=act.kind, detail=getattr(act, "name", None) or "final"))
        if act.kind == "final":
            return dict(answer=act.text, steps=step, log=log)
        if TOOLS.get(act.name, {}).get("destructive"):
            if on_destructive is None or not on_destructive(act):
                return dict(answer=f"HALTED: {act.name} needs approval", steps=step, log=log)
        result = TOOLS[act.name]["fn"](**act.args) if act.name in TOOLS else {"error": "unknown tool"}
        messages += [{"role": "assistant", "content": f"{act.name}({act.args})"},
                     {"role": "tool", "content": json.dumps(result)}]
    return dict(answer="STOPPED: max_steps", steps=max_steps, log=log)

out = run_agent_final("Which team has the highest budget per person?", react_planner)
print(json.dumps(out, indent=1)[:500])

{
 "answer": "sales has the highest budget per person ($13,333/person). Full ranking: sales $13,333, engineering $12,381, design $12,222",
 "steps": 3,
 "log": [
  {
   "step": 0,
   "kind": "tool",
   "detail": "get_team"
  },
  {
   "step": 1,
   "kind": "tool",
   "detail": "get_team"
  },
  {
   "step": 2,
   "kind": "tool",
   "detail": "get_team"
  },
  {
   "step": 3,
   "kind": "final",
   "detail": "final"
  }
 ]
}


## 6 — Exercises

1. **Add a tool.** Add `set_budget(team, amount)` that mutates `TEAMS`. Make it "destructive":
   route it through an `on_destructive` approval callback that prints the request and returns
   `False` (deny). Show the agent halts.
2. **Progress guard.** Extend `run_agent_guarded` to abort if the last 3 tool results are
   byte-identical even when the (tool, args) differ (a subtler no-progress loop).
3. **Planner for a new goal.** Write a planner for *"Can we afford to double the design team
   for a year?"* — it must look up design's budget + headcount, cash on hand, revenue, do the
   arithmetic, and answer yes/no with the number.
4. **Token budget.** Add a `max_input_tokens` guard to `run_agent_guarded` using `tiktoken` to
   count the running transcript; abort when it would exceed the budget on the next call.
5. **Bad-tool recovery.** Make a planner call `get_team("marketing")`, get `{"error": ...}`,
   and *recover* on the next step by trying a valid team. Show the loop tolerates tool errors.
6. **Cost vs steps.** Using `agent_cost`, find the number of steps at which an agent run
   exceeds \$0.10, and separately \$1.00. What does that imply for `max_steps` defaults?

In [11]:
# ---- Solution 1 ----
@tool("set_budget", "Set a team's monthly budget (DESTRUCTIVE).",
      {"team": {"type": "string"}, "amount": {"type": "number"}}, destructive=True)
def set_budget(team, amount):
    if team not in TEAMS: return {"error": "unknown team"}
    TEAMS[team]["monthly_budget"] = amount
    return {"ok": True, "team": team, "new_budget": amount}

def raise_design_planner(messages):
    n = len([m for m in messages if m["role"] == "tool"])
    if n == 0: return Action("tool", name="set_budget", args={"team": "design", "amount": 200000})
    return Action("final", text="done")

def deny(act):
    print(f"  APPROVAL REQUEST: {act.name}({act.args})  -> DENIED")
    return False

print(run_agent_final("Raise the design budget to 200k", raise_design_planner, on_destructive=deny))
print("design budget unchanged:", TEAMS["design"]["monthly_budget"])

  APPROVAL REQUEST: set_budget({'team': 'design', 'amount': 200000})  -> DENIED
{'answer': 'HALTED: set_budget needs approval', 'steps': 0, 'log': [{'step': 0, 'kind': 'tool', 'detail': 'set_budget'}]}
design budget unchanged: 110000


In [12]:
# ---- Solution 6 ----
def steps_to_exceed(limit):
    s = 1
    while agent_cost(s) < limit: s += 1
    return s
print(f"S6: exceeds $0.10 at {steps_to_exceed(0.10)} steps; exceeds $1.00 at {steps_to_exceed(1.00)} steps")
print("    -> default max_steps in the 6-12 range for most agents; only raise it with a hard")
print("       token/dollar budget guard also in place.")

S6: exceeds $0.10 at 13 steps; exceeds $1.00 at 47 steps
    -> default max_steps in the 6-12 range for most agents; only raise it with a hard
       token/dollar budget guard also in place.


### Solutions 2, 3, 4, 5 (sketch)

**S2:** keep a `deque(maxlen=3)` of `json.dumps(result)`; if `len(set(...)) == 1` and it's
full, abort with "no progress". Catches a model that varies its query but keeps getting the
same nothing back.

**S3:** planner steps: `get_team("design")` → `get_finance("cash_on_hand")` →
`get_finance("monthly_revenue")` → `calculator("6400000 / ((110000*2 - 110000) + (300000 -
... ))")` ... then `final`: "Doubling design adds $110k/month; net burn becomes $440k/month;
$6.4M / $440k ≈ 14.5 months — yes, affordable for a year."

**S4:** `enc = tiktoken.get_encoding("cl100k_base")`; before each planner call,
`if len(enc.encode(json.dumps(messages))) + est_response > max_input_tokens: abort`. Agents
die from context growth, not from step count alone.

**S5:** the recovery planner checks the last tool result: `if last and "error" in last:` pick
a different valid argument. A robust agent treats tool errors as observations to reason about,
not as crashes.

## Self-check quiz

1. In one sentence, what makes something an agent rather than a prompt-response system?
2. What does the agent loop feed back to the model after each tool call?
3. Give three distinct failure modes of an agent and one guard for each.
4. Why must input validation live inside the tool function, not in the prompt?
5. Why does an agent run cost grow super-linearly in the number of steps?
6. When should a tool call require human approval?
7. In the Anthropic tool-use API, how many messages do you use to return the results of three
   parallel tool calls, and of what role?

### Answer key

1. The model chooses the control flow — which action to take next and when to stop — instead
   of following a fixed pipeline you wrote.
2. The tool's result (the "observation"), appended to the conversation so the next model call
   can reason about it.
3. Infinite/repeated loop → repeated-action detection + `max_steps`; wrong or destructive
   action → tool-side validation + tool allowlist + human approval; runaway cost → tool-call
   budget + token-budget guard. (Also: hanging tool → per-call timeout.)
4. The model's tool arguments are untrusted input — it can hallucinate bad values or be
   prompt-injected. A prompt instruction is a suggestion; validation in the function is
   enforcement.
5. Each step is a fresh LLM call whose input is the entire growing transcript, so per-step
   input tokens rise roughly linearly with step number, making cumulative cost roughly
   quadratic.
6. When the action is destructive or irreversible and hard to recover from: deleting data,
   sending communications, spending money, deploying, modifying production.
7. One message, role `user`, containing all three `tool_result` content blocks. Splitting them
   across messages trains the model to stop making parallel calls.

## Where this goes next

- **Day 20 — Frameworks:** LangChain and LlamaIndex package the loop, tools, memory, and
  retrieval you just built. What each abstraction maps to, and how to choose.